# Hugging Face Fundamentals — Lesson 4: AutoTokenizer

> Learning material for **Hugging Face Fundamentals**. Companion to the lesson script `04_AutoTokenizer.py` (same content, runnable without Jupyter).

**Task ID:** HF-004  |  **Folder:** `04_AutoTokenizer`


## The problem

Every model family tokenizes differently:

- **BERT (WordPiece):** `tokenizer!` → `token` `##izer` `!`
- **GPT-2 (byte-level BPE):** → `token` `izer` `!`
- **T5 (SentencePiece):** → `▁token` `izer` `!` (the `▁` marks word starts)

Remembering which tokenizer each model needs is impossible. **AutoTokenizer** reads the model's config and loads the *right* one for you — that is the whole trick.


In [ ]:
from transformers import AutoTokenizer

bert = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2 = AutoTokenizer.from_pretrained("gpt2")
t5 = AutoTokenizer.from_pretrained("t5-small")

for name, tok in [("bert", bert), ("gpt2", gpt2), ("t5", t5)]:
    print(name, "->", tok.tokenize("tokenizer!"))


Same text, three vocabularies, three splitting rules — and AutoTokenizer picked each one automatically.

## Same text, different ids

The `encode / decode` round trip works for every family:


In [ ]:
for name, tok in [("bert", bert), ("gpt2", gpt2), ("t5", t5)]:
    ids = tok("tokenizer!")["input_ids"]
    print(f"{name:>4}  ids={ids}  back={tok.decode(ids)!r}")


Note GPT-2 has **no** `[CLS]`/`[SEP]`: causal models just read. T5 uses `</s>` as its separator. Different families, different conventions — AutoTokenizer handles them all.

## Padding, truncation, attention mask

For batches the tokenizer pads all texts to the same length. The **attention_mask** tells the model which positions are *real* words (1) and which are padding (0) — otherwise the model would learn from empty space:


In [ ]:
enc = bert(
    ["short", "a medium length sentence here", "a very long sentence that gets truncated"],
    padding=True, truncation=True, max_length=12, return_tensors="pt",
)
print("shape       :", tuple(enc["input_ids"].shape))
print("attention   :")
for mask in enc["attention_mask"]:
    print("   ", mask.tolist())


## Save and reload: tokenizers are just folders

A tokenizer is a small folder of files. Save it, share it, reload it — the same API as loading from the Hub:


In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    bert.save_pretrained(tmp)
    again = AutoTokenizer.from_pretrained(tmp)
    print("reloaded:", again.decode(again("works from a folder too")["input_ids"]))


## Try it yourself

1. Compare `cl100k_base`-style tokenizers: try `gpt2` vs `roberta-base`.
2. Tokenize the same long sentence with and without `truncation`.
3. Save a tokenizer, delete it, reload from the folder (offline!).

## Common pitfalls

- **Padding without `padding=True`** — batches break for variable lengths.
- **Forgetting `attention_mask`** — models get confused by the `[PAD]` ids.
- T5's tokenizer needs the `sentencepiece` package installed.

## Summary

- AutoTokenizer = the *right* tokenizer for any model, automatically.
- `padding`/`truncation`/`attention_mask` are batch essentials.
- Tokenizers are folders: save and reload them anywhere.

**Next lesson:** HF-005 — AutoModel.  |  Extra reading: `../resources/reference_links.md`
